In [1]:
import tkinter as tk
from tkinter import messagebox
from tkinter import Frame, Entry
import joblib
import numpy as np
import pandas as pd
from tensorflow import keras
import ast

In [2]:
Type_of_Loan_Encoder = joblib.load('encoder/Type_of_Loan_Encoder.pkl')
Credit_Mix_Encoder = joblib.load('encoder/Credit_Mix_Encoder.pkl')
Payment_of_Min_Amount_Encoder = joblib.load('encoder/Payment_of_Min_Amount_Encoder.pkl')
Payment_Behaviour_Encoder = joblib.load('encoder/Payment_Behaviour_Encoder.pkl')
model = keras.models.load_model('model/Credit_Score_Classification_Model.keras')

In [3]:
def preprocess_and_predict(data):
    converted_data = data.copy()

    try:
        converted_data[8] = Type_of_Loan_Encoder.transform([converted_data[8]])[0]
        converted_data[13] = Credit_Mix_Encoder.transform([[converted_data[13]]])[0][0]
        converted_data[17] = Payment_of_Min_Amount_Encoder.transform([converted_data[17]])[0]
        payment_behaviour = [[converted_data[20]]]
        encoded_value = Payment_Behaviour_Encoder.transform(payment_behaviour)
        converted_data[20] = encoded_value[0][0]

        input_data = pd.DataFrame([converted_data], columns=[
            'Age', 'Occupation', 'Annual_Income', 'Monthly_Inhand_Salary', 'Num_Bank_Accounts',
            'Num_Credit_Card', 'Interest_Rate', 'Num_of_Loan', 'Type_of_Loan', 'Delay_from_due_date',
            'Num_of_Delayed_Payment', 'Changed_Credit_Limit', 'Num_Credit_Inquiries', 'Credit_Mix',
            'Outstanding_Debt', 'Credit_Utilization_Ratio', 'Credit_History_Age', 'Payment_of_Min_Amount',
            'Total_EMI_per_month', 'Amount_invested_monthly', 'Payment_Behaviour', 'Monthly_Balance'
        ])

        predictions = model.predict(input_data)
        predicted_classes = np.argmax(predictions, axis=1)

        mapping = {0: 'Good', 1: 'Standard', 2: 'Poor'}
        string_predict = [mapping[pred] for pred in predicted_classes]

        return string_predict[0]
    
    except Exception as e:
        raise ValueError(f"Error in processing or prediction: {e}")

In [4]:
root = tk.Tk()
root.title("Credit Score Prediction")
root.iconbitmap("Icon/c.ico")
root.geometry("600x400+665+135")
root.minsize(600, 400)
root.maxsize(600, 400)
root.configure(bg="#e7e7e7")

input_label = tk.Label(root, text="Credit Score Prediction", font=("Arial", 12))
input_label.pack(pady=10)

data_input = tk.Text(root, height=10, width=70)
data_input.pack(pady=10)

output_label = tk.Label(root, text="", font=("Arial", 12))
output_label.pack(pady=20)

def on_predict_click():
    try:
        raw_data = data_input.get("1.0", tk.END).strip()
        data = ast.literal_eval(raw_data)
        if not isinstance(data, list):
            raise ValueError("Input must be a list.")
        
        if len(data) != 22:
            raise ValueError("Input data must contain 22 values.")
        
        prediction = preprocess_and_predict(data)
        output_label.config(text=f"Result : {prediction}")
    
    except Exception as e:
        messagebox.showerror("Error", f"An error occurred: {e}")

predict_button = tk.Button(root, text="Predict", command=on_predict_click, font=("Arial", 12), bg="lightblue")
predict_button.pack(pady=10)

root.mainloop()

1/1 [==============================] - ETA: 0s

C:\Users\watta\anaconda3\envs\python-cvcourse\lib\site-packages\sklearn\base.py:465: UserWarning: X does not have valid feature names, but OrdinalEncoder was fitted with feature names
  warnings.warn(
C:\Users\watta\anaconda3\envs\python-cvcourse\lib\site-packages\sklearn\base.py:465: UserWarning: X does not have valid feature names, but OrdinalEncoder was fitted with feature names
  warnings.warn(


1/1 [==============================] - 0s 153ms/step


In [5]:
 # [20.0, 10, 29741.12, 2151.702355, 8.0, 5.0, 31.0, 5.0, "Credit-Builder Loan, Auto Loan, Home Equity Loan, Home Equity Loan, and Student Loan", 18.0, 14.0, 7.56, 7.0, "Standard", 1284.84, 24.45386056, 207.0, "Yes", 253.2465227, 58.27554631, "High_spent_Medium_value_payments", 326.0449091]